In [1]:
from dotenv import load_dotenv
import os
from langchain_google_genai import(
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser

C:\Users\vansh_hxqeh4o\AppData\Local\Temp\ipykernel_1580\914739688.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings_hf = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
dimension = len(embeddings_hf.embed_query("test"))
print(f"Length of embedding vector: {dimension}")

Length of embedding vector: 384


In [4]:
#load pdf for chromadb
pdf_path="D:\GEN AI BASICS\RAG\Vector-database\Data\llama2-research-paper.pdf"
loader = PyPDFLoader(pdf_path)
pages = loader.load()
print(f"Number of pages in the PDF: {len(pages)}")

<>:2: SyntaxWarning: invalid escape sequence '\G'
<>:2: SyntaxWarning: invalid escape sequence '\G'
C:\Users\vansh_hxqeh4o\AppData\Local\Temp\ipykernel_1580\607216580.py:2: SyntaxWarning: invalid escape sequence '\G'
  pdf_path="D:\GEN AI BASICS\RAG\Vector-database\Data\llama2-research-paper.pdf"


Number of pages in the PDF: 77


In [5]:
#create a chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)
chunks=text_splitter.split_documents(pages)
print(f"Number of chunks created: {len(chunks)}")

Number of chunks created: 175


In [6]:
# Create a Chroma Vector Store
vectorstore = Chroma(
    collection_name="llama2-research-paper",
    embedding_function=embeddings_hf,
    persist_directory="./chroma_db_llama2",
    collection_metadata={
        "hnsw:space": "cosine"
    }
)

In [7]:
#add documents
document_ids = vectorstore.add_documents(chunks)
print(f"Number of documents added to the vector store: {len(document_ids)}")

Number of documents added to the vector store: 175


In [8]:
#creating a retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [10]:
#test retriever
query="what is the architecture of llama2?"
retrieved_docs = retriever.invoke(query)
for i,doc in enumerate(retrieved_docs,start=1):
    print(f"---- Retrieved Document {i} --")
    print(f"Content: {doc.page_content[:500]}...")  # Print first 500 characters of content
    print(f"Metadata: {doc.metadata}")

---- Retrieved Document 1 --
Content: guide¶ and code examples‖ to facilitate the safe deployment ofLlama 2 and Llama 2-Chat. More details of
our responsible release strategy can be found in Section 5.3.
The remainder of this paper describes our pretraining methodology (Section 2), fine-tuning methodology
(Section 3), approach to model safety (Section 4), key observations and insights (Section 5), relevant related
work (Section 6), and conclusions (Section 7).
‡https://ai.meta.com/resources/models-and-libraries/llama/
§We are delayi...
Metadata: {'producer': 'pdfTeX-1.40.25', 'author': '', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'page': 3, 'total_pages': 77, 'subject': '', 'title': '', 'moddate': '2023-07-20T00:30:36+00:00', 'keywords': '', 'creator': 'LaTeX with hyperref', 'page_label': '4', 'creationdate': '2023-07-20T00:30:36+00:00', 'source': 'D:\\GEN AI BASICS\\RAG\\Vector-database\\Data\\lla